# Ingest NutriAccess Food Desert Census Data

This notebook connects to the project S3 bucket, verifies the raw Census files exist, loads files, and saves processed outputs for later analysis.

This notebook is specific to the Censue data for the NutriAccess project.

2. Census Demographic Data¶

https://www.census.gov/data.html

Source: U.S. Census Bureau Analyze how income and demographics, such as income level, population density, Poverty rates and household composition are related to food desert prevalence.

Useful tables

- Income
- population density
- poverty rate

Download tables (filter for United States)

S1901: Income in the Past 12 Months
S1701: Poverty Status
DP05: Demographic and Housing Estimates

Purpose in the project: Analyze how income and demographics relate to food desert prevalence.

## Import Libaries 

In [1]:
import boto3
import pandas as pd 
import os
from dotenv import load_dotenv

## Connect to S3 bucket from local .env

In [2]:
load_dotenv()

AWS_REGION = os.getenv("AWS_REGION")
BUCKET = os.getenv("S3_BUCKET")
RAW_PREFIX = os.getenv("S3_PREFIX")
LOCAL_DATA_DIR = os.getenv("LOCAL_DATA_DIR")

if not BUCKET:
    raise ValueError("S3_BUCKET is not set in .env")

if not RAW_PREFIX:
    raise ValueError("S3_PREFIX is not set in .env")

s3 = boto3.client("s3", region_name=AWS_REGION)

## List every file that exists in the raw data folder

*Note: Many files will appear; some interact with each other and remain part of the same dataset. For example, geospatial datasets contain many interacting files*

In [3]:
# Use paginator in case there are many files
paginator = s3.get_paginator("list_objects_v2")

pages = paginator.paginate(
    Bucket=BUCKET,
    Prefix=RAW_PREFIX
)

files = []

for page in pages:
    for obj in page.get("Contents", []):
        files.append(obj["Key"])

print("Files found in rawData:")
for f in files:
    print(f)

Files found in rawData:
rawData/
rawData/ACSDP1Y2024.DP05-2026-03-13T140903.csv
rawData/ACSST1Y2024.S1701-2026-03-13T140807.csv
rawData/ACSST1Y2024.S1901-2026-03-13T140835.csv
rawData/Censue_Dataset/ACSDP5Y2023.DP05-Data.csv
rawData/Censue_Dataset/ACSST5Y2023.S1701-Data.csv
rawData/Censue_Dataset/ACSST5Y2023.S1901-Data.csv
rawData/FoodAccess/
rawData/FoodAccess/2019_Food_Access_Research_Atlas_Data/Food Access Research Atlas.csv
rawData/FoodAccess/2019_Food_Access_Research_Atlas_Data/ReadMe.csv
rawData/FoodAccess/2019_Food_Access_Research_Atlas_Data/VariableLookup.csv
rawData/FoodEnvironment/
rawData/FoodEnvironment/2025-food-environment-atlas-data/ReadMeFile2025.txt
rawData/FoodEnvironment/2025-food-environment-atlas-data/StateAndCountyData.csv
rawData/FoodEnvironment/2025-food-environment-atlas-data/VariableList.csv
rawData/PLACES__Local_Data_for_Better_Health,_County_Data,_2025_release_20260313.csv
rawData/geofabrik_NorCal/
rawData/geofabrik_NorCal/norcal-260312-free.shp/README
rawDa

# Load CSV files 

In [4]:
datasets = {}

skip_terms = ["readme", "variablelookup", "variablelist"]

for key in files:
    lower_key = key.lower()

    # Only load relevant CSVs
    if key.endswith(".csv") and not any(term in lower_key for term in skip_terms):
        path = f"s3://{BUCKET}/{key}"
        print(f"Loading {path}")

        try:
            # Clean dataset name
            name = key.split("/")[-1].replace(".csv", "")
            name = (
                name.lower()
                    .replace(" ", "_")
                    .replace(",", "")
                    .replace("-", "_")
                    .replace(".", "_")
            )

            # Load dataset
            df = pd.read_csv(path, low_memory=False)

            # Remove metadata row (ACS-specific)
            df = df.iloc[1:].reset_index(drop=True)

            datasets[name] = df

        except Exception as e:
            print(f"❌ Failed to load {key}: {e}")

print("\nDatasets loaded:")
print(list(datasets.keys()))

Loading s3://nutriaccess-data/rawData/ACSDP1Y2024.DP05-2026-03-13T140903.csv
Loading s3://nutriaccess-data/rawData/ACSST1Y2024.S1701-2026-03-13T140807.csv
Loading s3://nutriaccess-data/rawData/ACSST1Y2024.S1901-2026-03-13T140835.csv
Loading s3://nutriaccess-data/rawData/Censue_Dataset/ACSDP5Y2023.DP05-Data.csv
Loading s3://nutriaccess-data/rawData/Censue_Dataset/ACSST5Y2023.S1701-Data.csv
Loading s3://nutriaccess-data/rawData/Censue_Dataset/ACSST5Y2023.S1901-Data.csv
Loading s3://nutriaccess-data/rawData/FoodAccess/2019_Food_Access_Research_Atlas_Data/Food Access Research Atlas.csv
Loading s3://nutriaccess-data/rawData/FoodEnvironment/2025-food-environment-atlas-data/StateAndCountyData.csv
Loading s3://nutriaccess-data/rawData/PLACES__Local_Data_for_Better_Health,_County_Data,_2025_release_20260313.csv

Datasets loaded:
['acsdp1y2024_dp05_2026_03_13t140903', 'acsst1y2024_s1701_2026_03_13t140807', 'acsst1y2024_s1901_2026_03_13t140835', 'acsdp5y2023_dp05_data', 'acsst5y2023_s1701_data', 

## Assign Census CSV 

In [5]:
dp05_df = datasets['acsdp5y2023_dp05_data']
s1701_df = datasets['acsst5y2023_s1701_data']
s1901_df = datasets['acsst5y2023_s1901_data']

# DP05 Demographic and Housing Estimates

In [6]:
# Select relevant columns
dp05_clean = dp05_df[[
    "GEO_ID",
    "NAME",
    "DP05_0001E",  # Total population
    "DP05_0002E",  # Male population
    "DP05_0003E",  # Female population
    "DP05_0018E"   # Median age
]].copy()

# Rename columns for clarity
dp05_clean.columns = [
    "geo_id",
    "county_name",
    "total_population",
    "male_population",
    "female_population",
    "median_age"
]

# Convert numeric columns
numeric_cols = [
    "total_population",
    "male_population",
    "female_population",
    "median_age"
]

for col in numeric_cols:
    dp05_clean[col] = pd.to_numeric(dp05_clean[col], errors="coerce")

# Split county and state
dp05_clean[["county", "state"]] = dp05_clean["county_name"].str.split(", ", expand=True)

# Remove "County" from county names
dp05_clean["county"] = dp05_clean["county"].str.replace(" County", "", regex=False)

# Standardize for merging (optional but recommended)
dp05_clean["county"] = dp05_clean["county"].str.lower()
dp05_clean["state"] = dp05_clean["state"].str.lower()

# Preview final cleaned dataset
dp05_clean.head()

#Cleaned Census DP05 demographic data by removing metadata rows, selecting key features, converting data types, and standardizing geographic identifiers for downstream merging.

,geo_id,county_name,total_population,male_population,female_population,median_age,county,state
0,0500000US01001,"Autauga County, Alabama",59285,28669,30616,39.2,autauga,alabama
1,0500000US01003,"Baldwin County, Alabama",239945,117316,122629,43.7,baldwin,alabama
2,0500000US01005,"Barbour County, Alabama",24757,12906,11851,40.7,barbour,alabama
3,0500000US01007,"Bibb County, Alabama",22152,11824,10328,41.3,bibb,alabama
4,0500000US01009,"Blount County, Alabama",59292,29934,29358,40.9,blount,alabama


# S1901: Income in the Past 12 Months

In [7]:
s1901_clean = s1901_df[[
    "GEO_ID",
    "NAME",
    "S1901_C01_012E"  # Median household income
]].copy()

s1901_clean.columns = [
    "geo_id",
    "county_name",
    "median_income"
]

s1901_clean["median_income"] = pd.to_numeric(s1901_clean["median_income"], errors="coerce")

s1901_clean[["county", "state"]] = s1901_clean["county_name"].str.split(", ", expand=True)
s1901_clean["county"] = s1901_clean["county"].str.replace(" County", "", regex=False).str.lower()
s1901_clean["state"] = s1901_clean["state"].str.lower()

# Preview final cleaned dataset
s1901_clean.head()

,geo_id,county_name,median_income,county,state
0,0500000US01001,"Autauga County, Alabama",69841.0,autauga,alabama
1,0500000US01003,"Baldwin County, Alabama",75019.0,baldwin,alabama
2,0500000US01005,"Barbour County, Alabama",44290.0,barbour,alabama
3,0500000US01007,"Bibb County, Alabama",51215.0,bibb,alabama
4,0500000US01009,"Blount County, Alabama",61096.0,blount,alabama


# S1701: Poverty Status

In [8]:
# Select key columns
s1701_clean = s1701_df[[
    "GEO_ID",
    "NAME",
    "S1701_C03_001E"  # Poverty rate (%)
]].copy()

s1701_clean.columns = [
    "geo_id",
    "county_name",
    "poverty_rate"
]

# Convert
s1701_clean["poverty_rate"] = pd.to_numeric(s1701_clean["poverty_rate"], errors="coerce")

# Split county/state (same as DP05)
s1701_clean[["county", "state"]] = s1701_clean["county_name"].str.split(", ", expand=True)
s1701_clean["county"] = s1701_clean["county"].str.replace(" County", "", regex=False).str.lower()
s1701_clean["state"] = s1701_clean["state"].str.lower()

# Preview final cleaned dataset
s1701_clean.head()

,geo_id,county_name,poverty_rate,county,state
0,0500000US01001,"Autauga County, Alabama",10.7,autauga,alabama
1,0500000US01003,"Baldwin County, Alabama",10.5,baldwin,alabama
2,0500000US01005,"Barbour County, Alabama",21.9,barbour,alabama
3,0500000US01007,"Bibb County, Alabama",20.5,bibb,alabama
4,0500000US01009,"Blount County, Alabama",14.1,blount,alabama


## Release Resources

In [9]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>

In [10]:
%%javascript

try {
    Jupyter.notebook.save_checkpoint();
    Jupyter.notebook.session.delete();
}
catch(err) {
    // NoOp
}

<IPython.core.display.Javascript object>